In [10]:
import httpx
import json
import pandas as pd
import typing


# Scratch Functions

In [25]:
# All the base_urls with routes are always paginated and have results field into it.

def get_paginated_base_url_results(url: str) -> list:
    """This function gets all the paginated result URLs"""

    response = httpx.get(url)
    response_data: dict = response.json()
    url_results: list = response_data['results']

    if 'next' in response_data and response_data['next']:
        paginated_url_result: list = get_paginated_base_url_results(
            response_data['next'])
        url_results += paginated_url_result

    return url_results


get_paginated_base_url_results("https://pokeapi.co/api/v2/version-group/")

[{'name': 'red-blue', 'url': 'https://pokeapi.co/api/v2/version-group/1/'},
 {'name': 'yellow', 'url': 'https://pokeapi.co/api/v2/version-group/2/'},
 {'name': 'gold-silver', 'url': 'https://pokeapi.co/api/v2/version-group/3/'},
 {'name': 'crystal', 'url': 'https://pokeapi.co/api/v2/version-group/4/'},
 {'name': 'ruby-sapphire',
  'url': 'https://pokeapi.co/api/v2/version-group/5/'},
 {'name': 'emerald', 'url': 'https://pokeapi.co/api/v2/version-group/6/'},
 {'name': 'firered-leafgreen',
  'url': 'https://pokeapi.co/api/v2/version-group/7/'},
 {'name': 'diamond-pearl',
  'url': 'https://pokeapi.co/api/v2/version-group/8/'},
 {'name': 'platinum', 'url': 'https://pokeapi.co/api/v2/version-group/9/'},
 {'name': 'heartgold-soulsilver',
  'url': 'https://pokeapi.co/api/v2/version-group/10/'},
 {'name': 'black-white', 'url': 'https://pokeapi.co/api/v2/version-group/11/'},
 {'name': 'colosseum', 'url': 'https://pokeapi.co/api/v2/version-group/12/'},
 {'name': 'xd', 'url': 'https://pokeapi.co/

In [21]:
response = httpx.get('https://pokeapi.co/api/v2/version-group/7/')

response.json()

{'generation': {'name': 'generation-iii',
  'url': 'https://pokeapi.co/api/v2/generation/3/'},
 'id': 7,
 'move_learn_methods': [{'name': 'level-up',
   'url': 'https://pokeapi.co/api/v2/move-learn-method/1/'},
  {'name': 'egg', 'url': 'https://pokeapi.co/api/v2/move-learn-method/2/'},
  {'name': 'tutor', 'url': 'https://pokeapi.co/api/v2/move-learn-method/3/'},
  {'name': 'machine',
   'url': 'https://pokeapi.co/api/v2/move-learn-method/4/'}],
 'name': 'firered-leafgreen',
 'order': 11,
 'pokedexes': [{'name': 'kanto',
   'url': 'https://pokeapi.co/api/v2/pokedex/2/'}],
 'regions': [{'name': 'kanto', 'url': 'https://pokeapi.co/api/v2/region/1/'}],
 'versions': [{'name': 'firered',
   'url': 'https://pokeapi.co/api/v2/version/10/'},
  {'name': 'leafgreen', 'url': 'https://pokeapi.co/api/v2/version/11/'}]}

In [7]:
url = 'https://pokeapi.co/api/v2/pokemon/113/'

response_data = httpx.get(url).json()

response_data

{'abilities': [{'ability': {'name': 'natural-cure',
    'url': 'https://pokeapi.co/api/v2/ability/30/'},
   'is_hidden': False,
   'slot': 1},
  {'ability': {'name': 'serene-grace',
    'url': 'https://pokeapi.co/api/v2/ability/32/'},
   'is_hidden': False,
   'slot': 2},
  {'ability': {'name': 'healer',
    'url': 'https://pokeapi.co/api/v2/ability/131/'},
   'is_hidden': True,
   'slot': 3}],
 'base_experience': 395,
 'cries': {'latest': 'https://raw.githubusercontent.com/PokeAPI/cries/main/cries/pokemon/latest/113.ogg',
  'legacy': 'https://raw.githubusercontent.com/PokeAPI/cries/main/cries/pokemon/legacy/113.ogg'},
 'forms': [{'name': 'chansey',
   'url': 'https://pokeapi.co/api/v2/pokemon-form/113/'}],
 'game_indices': [{'game_index': 40,
   'version': {'name': 'red', 'url': 'https://pokeapi.co/api/v2/version/1/'}},
  {'game_index': 40,
   'version': {'name': 'blue', 'url': 'https://pokeapi.co/api/v2/version/2/'}},
  {'game_index': 40,
   'version': {'name': 'yellow',
    'url': '

In [8]:

key_rejections_dict = {
    'url': 'name',
    'symbol_icon': 'name_icon',
    'game_index': 'generation'
}

def clean_unwanted_keys(data_dict, key_rejections_dict):
    replaced_dict = data_dict
    for key in key_rejections_dict:
        if key in data_dict:
            # key present in the data_dict.
            replacement_key = key_rejections_dict[key]
            if replacement_key in data_dict:
                # Replacement key is also present.
                replaced_dict = data_dict[replacement_key]

                if isinstance(replaced_dict, dict):
                    # Checking if the replacement data is again dictionary.
                    replaced_dict = clean_unwanted_keys(replaced_dict, key_rejections_dict)
            else:
                break
    return replaced_dict


def walk_nested_data(data, key_rejections_dict):
    cleaned_data = None
    if isinstance(data, dict):
        cleaned_data = {}
        cleaned_data = clean_unwanted_keys(data, key_rejections_dict)
        if not isinstance(cleaned_data, str):
            for sub_data in cleaned_data:
                cleaned_data[sub_data] = walk_nested_data(
                    data[sub_data], key_rejections_dict
                )
    elif isinstance(data, list):
        cleaned_data = []
        for sub_data in data:
            cleaned_data.append(walk_nested_data(
                sub_data, key_rejections_dict))
    else:
        return data
        
    return cleaned_data


nested_response_data = walk_nested_data(response_data, key_rejections_dict)

        

In [9]:
nested_response_data

{'abilities': [{'ability': 'natural-cure', 'is_hidden': False, 'slot': 1},
  {'ability': 'serene-grace', 'is_hidden': False, 'slot': 2},
  {'ability': 'healer', 'is_hidden': True, 'slot': 3}],
 'base_experience': 395,
 'cries': {'latest': 'https://raw.githubusercontent.com/PokeAPI/cries/main/cries/pokemon/latest/113.ogg',
  'legacy': 'https://raw.githubusercontent.com/PokeAPI/cries/main/cries/pokemon/legacy/113.ogg'},
 'forms': ['chansey'],
 'game_indices': [{'game_index': 40, 'version': 'red'},
  {'game_index': 40, 'version': 'blue'},
  {'game_index': 40, 'version': 'yellow'},
  {'game_index': 113, 'version': 'gold'},
  {'game_index': 113, 'version': 'silver'},
  {'game_index': 113, 'version': 'crystal'},
  {'game_index': 113, 'version': 'ruby'},
  {'game_index': 113, 'version': 'sapphire'},
  {'game_index': 113, 'version': 'emerald'},
  {'game_index': 113, 'version': 'firered'},
  {'game_index': 113, 'version': 'leafgreen'},
  {'game_index': 113, 'version': 'diamond'},
  {'game_index

In [25]:
pd.DataFrame(nested_response_data['game_indices'])

,game_index,version
0,180,red
1,180,blue
2,180,yellow
3,6,gold
4,6,silver
5,6,crystal
6,6,ruby
7,6,sapphire
8,6,emerald
9,6,firered


In [55]:
unnested_moves = []
for idx in range(len(nested_response_data['moves'])):
    move_dict = nested_response_data['moves'][idx]

    move_name = nested_response_data['moves'][idx]['move']
    for version_detail in nested_response_data['moves'][idx]['version_group_details']:
        version_detail.update({'move': move_name})
        unnested_moves.append(version_detail)

move_data_df = pd.DataFrame(unnested_moves)


move_data_df[move_data_df['version_group'] ==
             'firered-leafgreen'].sort_values(['level_learned_at', 'move_learn_method', 'order']).reset_index(drop=True)

,level_learned_at,move_learn_method,order,version_group,move
0,0,machine,NaN,firered-leafgreen,cut
1,0,machine,NaN,firered-leafgreen,fly
2,0,machine,NaN,firered-leafgreen,roar
3,0,machine,NaN,firered-leafgreen,flamethrower
4,0,machine,NaN,firered-leafgreen,hyper-beam
5,0,machine,NaN,firered-leafgreen,strength
6,0,machine,NaN,firered-leafgreen,earthquake
7,0,machine,NaN,firered-leafgreen,dig
8,0,machine,NaN,firered-leafgreen,toxic
9,0,machine,NaN,firered-leafgreen,double-team


# Extractor Tool Class

In [29]:
import httpx
import typing


class PokeApiExtractorTools:

    def __init__(self):
        # This map helps in cleaning up unuwanted keys from the response.
        self.KEY_REJECTION_MAP = {
            'url': 'name',
            'symbol_icon': 'name_icon',
            'game_index': 'generation'
        }
        self.VERSION_GROUP_TO_VERSION_MAP = self.get_version_group_to_version_map()

    def get_paginated_base_url_results(self, url: str) -> list:
        """This function gets all the paginated result URLs"""
        response = httpx.get(url)
        response_data: dict = response.json()
        url_results: list = response_data['results']

        if 'next' in response_data and response_data['next']:
            paginated_url_result: list = get_paginated_base_url_results(
                response_data['next'])
            url_results += paginated_url_result

        return url_results

    def get_version_group_to_version_map(self) -> dict:
        """This function gets the version_group_to_version_map details"""
        version_group_to_version_map: dict = {}

        versions_group_url: str = "https://pokeapi.co/api/v2/version-group/"
        version_group_url_list: list = self.get_paginated_base_url_results(
            versions_group_url
        )

        for url_name_map in version_group_url_list:
            version_result =  httpx.get(
                url_name_map['url']
            ).json()
            if 'versions' in version_result:
                version_group_to_version_map[url_name_map['name']] = [
                    version_name_map['name'] 
                    for version_name_map in version_result['versions']
                ]

        return version_group_to_version_map


    def __clean_unwanted_keys(self, data_dict: dict) -> dict:
        """This function cleans unwanted keys from the data."""
        replaced_dict = data_dict
        for key in self.KEY_REJECTION_MAP:
            if key in data_dict:
                # key present in the data_dict.
                replacement_key = self.KEY_REJECTION_MAP[key]
                if replacement_key in data_dict:
                    # Replacement key is also present.
                    replaced_dict = data_dict[replacement_key]

                    if isinstance(replaced_dict, dict):
                        # Checking if the replacement data is again dictionary.
                        replaced_dict = self.__clean_unwanted_keys(replaced_dict)
                else:
                    break
        return replaced_dict


    def walk_nested_data(self, data: typing.Union[dict, list, str]) -> dict:
        """This function walks across the entire data and cleans the unwanted keys"""
        cleaned_data = None
        if isinstance(data, dict):
            cleaned_data = {}
            cleaned_data = self.__clean_unwanted_keys(data)
            if not isinstance(cleaned_data, str):
                for sub_data in cleaned_data:
                    cleaned_data[sub_data] = walk_nested_data(
                        data[sub_data], key_rejections_dict
                    )
        elif isinstance(data, list):
            cleaned_data = []
            for sub_data in data:
                cleaned_data.append(walk_nested_data(
                    sub_data, key_rejections_dict))
        else:
            return data

        return cleaned_data

In [31]:
pokeapi_extractor_tools = PokeApiExtractorTools()

# Pokemon endpoint Scratch

In [34]:
url = 'https://pokeapi.co/api/v2/pokemon/113/'

response_data = httpx.get(url).json()

In [35]:
response_data.keys()

dict_keys(['abilities', 'base_experience', 'cries', 'forms', 'game_indices', 'height', 'held_items', 'id', 'is_default', 'location_area_encounters', 'moves', 'name', 'order', 'past_abilities', 'past_stats', 'past_types', 'species', 'sprites', 'stats', 'types', 'weight'])

Looking at the data, it would be better to create multiple tables for stats. This makes this data a little better.

In [54]:
url = "https://pokeapi.co/api/v2/location-area/2"
httpx.get(url).json()

{'encounter_method_rates': [{'encounter_method': {'name': 'old-rod',
    'url': 'https://pokeapi.co/api/v2/encounter-method/2/'},
   'version_details': [{'rate': 25,
     'version': {'name': 'diamond',
      'url': 'https://pokeapi.co/api/v2/version/12/'}},
    {'rate': 25,
     'version': {'name': 'pearl',
      'url': 'https://pokeapi.co/api/v2/version/13/'}},
    {'rate': 25,
     'version': {'name': 'platinum',
      'url': 'https://pokeapi.co/api/v2/version/14/'}}]},
  {'encounter_method': {'name': 'good-rod',
    'url': 'https://pokeapi.co/api/v2/encounter-method/3/'},
   'version_details': [{'rate': 50,
     'version': {'name': 'diamond',
      'url': 'https://pokeapi.co/api/v2/version/12/'}},
    {'rate': 50,
     'version': {'name': 'pearl',
      'url': 'https://pokeapi.co/api/v2/version/13/'}},
    {'rate': 50,
     'version': {'name': 'platinum',
      'url': 'https://pokeapi.co/api/v2/version/14/'}}]},
  {'encounter_method': {'name': 'super-rod',
    'url': 'https://pokeap